# Summary
Script to run against **one** cleaned file (CNAF, MSA, CNOUS or FC) that has already gone through
its own cleaning/formatting notebook.

This notebook:
- loads a single export, no merging between sources nor with the previous year,
- sets the default column values: exercice_id, uuid_doc, zrr, qpv, a_valider, refuser, created_at, updated_at,
- generates unique codes and assigns them to the "id_psp" column,
- writes the enriched rows to a **new** CSV, leaving the input file untouched,
- appends the codes it just used to `EXISTING_CODES_PATHFILE_2026`.

`EXISTING_CODES_PATHFILE_2026` is a single column ("code") CSV holding every code already handed
out for this exercice. It is read before generating anything and rewritten at the end, which is
what keeps codes unique across the successive runs of this notebook (one per partner file, one
per CNOUS wave, ...).

⚠️ It deduplicates **codes**, never **beneficiaries**: running it twice on the same people hands
them two codes. Partner files are one-shot exports so the question does not arise, but the `FC`
source is a query against a live table — it carries its own write-back step to mark whoever has
already been served. See [franceconnect/README.md](franceconnect/README.md).

The step itself lives in [generate_codes_lib.py](generate_codes_lib.py), on top of the generic
code drawing and bookkeeping of [../../utils/codes_utils.py](../../utils/codes_utils.py). This
notebook and the FranceConnect cron ([franceconnect/run_fc_pipeline.sh](franceconnect/run_fc_pipeline.sh),
through `fc_pipeline.py codes`) call the very same function, so running by hand and running
automatically cannot drift apart.

Pick the file to process with the `SOURCE` variable in the configuration cell below.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# generate_codes_lib sits next to this notebook and imports utils.data_utils, which lives at
# the data/ root: make that root importable, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

import generate_codes_lib as lib

load_dotenv()

In [ ]:
# Which cleaned file to process: see generate_codes_lib.SOURCE_INPUT_ENV_VAR for the full
# list ('CNAF', 'CNAF_AAH_AEEH', 'MSA', 'MSA_AAH_AEEH', 'CNOUS' or 'FC').
SOURCE = 'CNAF'

input_filepath = os.environ[lib.SOURCE_INPUT_ENV_VAR[SOURCE]]
existing_codes_filepath = os.environ['EXISTING_CODES_PATHFILE_2026']

# The input file is never modified in place: rows enriched with their default columns and
# their id_psp go to a dated file sitting next to it.
output_filepath = lib.dated_output_path(input_filepath, SOURCE)

print(f"input:          {input_filepath}")
print(f"existing codes: {existing_codes_filepath}")
print(f"output:         {output_filepath}")

In [ ]:
# The whole step, in generate_codes_lib.generate_codes_for_file: read the cleaned file, add
# the production default columns, draw one brand new code per row, write the dated output,
# then track the codes just handed out. The code list is only rewritten once the output file
# is safely on disk, so a run that dies midway does not burn codes it never handed out.
stats = lib.generate_codes_for_file(input_filepath, output_filepath, existing_codes_filepath)

for key, value in stats.items():
    print(f"{key}: {value}")